<div style="background:linear-gradient(135deg,#0b1024 0%,#1e1b4b 52%,#312e81 100%);padding:56px 44px;border-radius:24px;text-align:center;border:1px solid #6366f1;box-shadow:0 14px 42px rgba(30,27,75,.35);">
<h1 style="color:#f8fafc;margin:0;font-size:2.4em;">Perakende Talep Tahmini</h1>
<p style="color:#c7d2fe;margin:14px 0 0;font-size:1.15em;">Ürün bazında talep oluşumu ve KG / ADT miktar tahmini · Uçtan uca veri hikâyesi</p>
<p style="color:#a5b4fc;margin:12px 0 0;">EDA → DataPrep → Model → Deployment → Monitoring</p>
</div>

<div style="background:#101827;border-left:5px solid #34d399;padding:22px 28px;border-radius:12px;margin-top:18px;">
<h3 style="color:#a7f3d0;margin:0 0 8px;">Notebook kullanım sözleşmesi</h3>
<p style="color:#d1fae5;margin:0;">Her adım <b>amaç → çalıştırılmış kod/görsel → kanıta dayalı Türkçe karar ve sonraki agent'a handoff</b> düzenindedir. Bu notebook üretim scriptlerinin yerine geçmez; onların ürettiği çıktıları anlaşılır ve izlenebilir bir hikâyede birleştirir.</p>
</div>

In [1]:
from pathlib import Path
import json
import sys
import warnings
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import Markdown, display

warnings.filterwarnings('ignore')
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
project_root_text = str(PROJECT_ROOT)
sys.path[:] = [entry for entry in sys.path if entry != project_root_text]
sys.path.insert(0, project_root_text)
RAW_DATA = PROJECT_ROOT / 'data' / 'data.csv'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
MODEL_READY_DIR = PROJECT_ROOT / 'data' / 'model_ready'
REPORTS_DIR = PROJECT_ROOT / 'reports'
BUNDLE_DIR = PROJECT_ROOT / 'models' / 'demand_forecasting_bundle'

PALETTE = {'indigo':'#6366f1','cyan':'#22d3ee','green':'#34d399','amber':'#fbbf24','rose':'#fb7185','text':'#e5e7eb'}
print(f'Proje kökü: {PROJECT_ROOT}')
print(f'Ham veri bulundu: {RAW_DATA.exists()}')

Proje kökü: /Users/ahmet/Desktop/proje
Ham veri bulundu: True


<div style="background:linear-gradient(90deg,#172554,#1e3a8a);border-left:6px solid #60a5fa;padding:18px 26px;border-radius:10px;margin:28px 0 12px;"><h2 style="color:#dbeafe;margin:0;">Bölüm 1 · İş Problemi ve Tahmin Sözleşmesi</h2><p style="color:#93c5fd;margin:7px 0 0;">Sorumlu agent: EDA Expert · Ürün, hedef tarih, birim ve kullanım sınırı</p></div>

### Soru
Belirli bir ürün için kullanıcının seçtiği gelecek günde talep oluşacak mı; oluşursa o gün kaç `KG` veya `ADT` gerekir?

- `forecast_origin`: yalnızca bu güne kadar bilinen bilgiler
- `target_date`: kullanıcının Streamlit tarih seçiciden belirlediği tek gelecek gün
- `lead_days`: hedef tarihin forecast origin'den kaç gün ileride olduğu
- Hedefler: `demand_occurs` ve `target_demand`
- `KG` ile `ADT` asla ortak bir hedef veya hata metriğinde birleştirilmez.

> **Agent kararı:** İlk sürüm son bilinen veri gününden 1–180 gün ileri seçilebilir tarihi destekler. Satışsız gün politikası ve satın alma sınırı EDA/DataPrep handoff'larında belgelenir.

In [2]:
if not RAW_DATA.exists():
    raise FileNotFoundError(f'Ham veri bulunamadı: {RAW_DATA}')
raw = pd.read_csv(RAW_DATA, sep=';', encoding='utf-8-sig')
display(raw.head())
print(f'Satır: {len(raw):,} | Kolon: {len(raw.columns)}')
print('Kolonlar:', list(raw.columns))

,satıs_tarıhı,urun_ıd,urun_ad,satılan_mıktar
0,1.01.2025,10002013,DANA KONTRAFILE BIFTEK KG,"0,384 KG"
1,1.01.2025,10002014,DANA DOS KIYMALIK ET KG,"15,174 KG"
2,1.01.2025,10002015,DANA TASKEBAPLIK BUT ETI KG,"2,738 KG"
3,1.01.2025,10002025,DANA SOTELIK BUT ETI KG,"6,832 KG"
4,1.01.2025,10002038,DANA ANTRIKOT BIFTEK KG,"0,394 KG"


Satır: 183,184 | Kolon: 4
Kolonlar: ['satıs_tarıhı', 'urun_ıd', 'urun_ad', 'satılan_mıktar']


<div style="background:linear-gradient(90deg,#0c2b36,#164e63);border-left:6px solid #22d3ee;padding:18px 26px;border-radius:10px;margin:28px 0 12px;"><h2 style="color:#cffafe;margin:0;">Bölüm 2 · Veri Anlama ve EDA</h2><p style="color:#67e8f9;margin:7px 0 0;">Sorumlu agent: EDA Expert · Veri kapsama, kalite, ürün portföyü ve zaman dinamiği</p></div>

In [3]:
# Türkçe tarih ve miktar/birim ayrıştırma yalnızca EDA görünümü içindir.
# Kalıcı temizleme ve panel üretimi DataPrep Expert'in onaylı scriptiyle yapılır.
eda = raw.copy()
eda['date'] = pd.to_datetime(eda['satıs_tarıhı'], dayfirst=True, errors='coerce')
parsed = eda['satılan_mıktar'].str.extract(r'^\s*([0-9][0-9.,]*)\s*([A-Za-z]+)\s*$')
eda['unit'] = parsed[1].str.upper()
def parse_tr_number(value):
    value = str(value).strip()
    if ',' in value and '.' in value:
        return float(value.replace('.', '').replace(',', '.'))
    # Kaynak veri Türkçe biçimdedir: virgül ondalık, nokta binlik ayıracıdır.
    return float(value.replace('.', '').replace(',', '.'))
eda['quantity'] = parsed[0].map(lambda x: parse_tr_number(x) if pd.notna(x) else np.nan)
profile = pd.DataFrame({
    'Metrik': ['Gözlem', 'Ürün', 'Tarih başlangıcı', 'Tarih bitişi', 'Tarih parse hatası', 'Miktar parse hatası'],
    'Değer': [len(eda), eda['urun_ıd'].nunique(), eda['date'].min(), eda['date'].max(), eda['date'].isna().sum(), eda['quantity'].isna().sum()]
})
display(profile)
display(eda['unit'].value_counts(dropna=False).rename_axis('Birim').reset_index(name='Satır sayısı'))

,Metrik,Değer
0,Gözlem,183184
1,Ürün,713
2,Tarih başlangıcı,2023-01-01 00:00:00
3,Tarih bitişi,2026-07-21 00:00:00
4,Tarih parse hatası,0
5,Miktar parse hatası,0


,Birim,Satır sayısı
0,KG,93279
1,ADT,89905


In [4]:
daily_unit = eda.dropna(subset=['date', 'quantity', 'unit']).groupby(['date', 'unit'], as_index=False)['quantity'].sum()
fig = px.line(daily_unit, x='date', y='quantity', color='unit',
              title='Birim Bazında Günlük Toplam Satış Talebi',
              labels={'date':'Tarih','quantity':'Toplam miktar','unit':'Birim'},
              color_discrete_map={'KG': PALETTE['cyan'], 'ADT': PALETTE['amber']})
fig.update_layout(template='plotly_dark', paper_bgcolor='#111827', plot_bgcolor='#111827')
fig.show()

history = eda.groupby('urun_ıd')['date'].agg(['min', 'max', 'nunique']).reset_index().rename(columns={'urun_ıd': 'product_id'})
history.columns = ['product_id', 'first_sale', 'last_sale', 'active_days']
display(history['active_days'].describe().to_frame('Aktif satış günü'))

,Aktif satış günü
count,713.000000
mean,256.920056
std,346.124029
min,1.000000
25%,16.000000
50%,76.000000
75%,372.000000
max,1297.000000


### Özel tarihlerin talebe etkisi

Resmî/dinî tatiller, Ramazan, bayramdan önceki ve sonraki üç gün, hafta sonu ile
MEB okul/ara/yarıyıl/yaz tatilleri aynı sürümlü Türkiye takviminden gelir. Aşağıdaki
oranlarda `1`, normal gün seviyesidir; `1` üzeri artış, altı azalış gösterir.
Bu bölüm betimseldir; model katkısı ayrıca validation ablation ile ölçülür.

In [5]:
calendar_impact_path = REPORTS_DIR / 'csv' / 'calendar_demand_impact.csv'
calendar_impact = pd.read_csv(calendar_impact_path)
display(calendar_impact)

plot_frame = calendar_impact[
    calendar_impact['calendar_context'].ne('ordinary_day')
].melt(
    id_vars=['unit', 'calendar_context'],
    value_vars=[
        'positive_rate_ratio_vs_ordinary',
        'mean_demand_ratio_vs_ordinary',
    ],
    var_name='metric',
    value_name='ratio',
)
fig = px.bar(
    plot_frame,
    x='calendar_context',
    y='ratio',
    color='metric',
    facet_col='unit',
    barmode='group',
    title='Normal Güne Göre Özel Takvim Talep Etkisi',
)
fig.add_hline(y=1.0, line_dash='dash', line_color=PALETTE['rose'])
fig.update_xaxes(tickangle=35)
fig.update_layout(
    template='plotly_dark',
    paper_bgcolor='#111827',
    plot_bgcolor='#111827',
)
fig.show()

,unit,calendar_context,rows,products,positive_rate,mean_daily_demand,median_daily_demand,total_demand,ordinary_positive_rate,ordinary_mean_daily_demand,positive_rate_ratio_vs_ordinary,mean_demand_ratio_vs_ordinary
0,ADT,ordinary_day,93288,393,0.317458,1.131710,0.0,105575.000,0.317458,1.131710,1.000000,1.000000
1,ADT,post_holiday_3d,20281,398,0.322321,1.090282,0.0,22112.000,0.317458,1.131710,1.015321,0.963393
2,ADT,pre_holiday_3d,19847,397,0.347861,1.288507,0.0,25573.000,0.317458,1.131710,1.095771,1.138548
3,ADT,public_holiday,13244,398,0.339777,1.229915,0.0,16289.000,0.317458,1.131710,1.070305,1.086776
4,ADT,ramadan_nonholiday,21224,363,0.356342,1.607944,0.0,34127.000,0.317458,1.131710,1.122486,1.420809
5,ADT,religious_special_day,2700,346,0.335926,1.157778,0.0,3126.000,0.317458,1.131710,1.058175,1.023034
6,ADT,school_midterm_break,5994,303,0.346513,1.374041,0.0,8236.000,0.317458,1.131710,1.091525,1.214127
7,ADT,school_semester_break,11803,331,0.372194,1.402864,0.0,16558.000,0.317458,1.131710,1.172419,1.239596
8,ADT,school_summer_break,41843,364,0.339937,1.171068,0.0,49001.000,0.317458,1.131710,1.070811,1.034777
9,ADT,weekend_nonholiday,33441,392,0.394276,1.617505,0.0,54091.000,0.317458,1.131710,1.241981,1.429257


### Bursa hava durumu ve talep ilişkisi

Gerçekleşen ERA5 hava değerleri bu bölümde **yalnız betimsel EDA** amacıyla
satışla eşlenir. Oranın `1` üzerinde olması ilgili hava koşulundaki toplam günlük
talebin diğer günlerden yüksek olduğunu gösterir; nedensellik göstermez. Mevsim,
ürün portföyü, fiyat, kampanya ve stok durumu bu ilişkiyi karıştırabilir.

Geleceğin gerçekleşmiş havası tahmin anında bilinmediği için aşağıdaki gerçekleşen
değerler model feature'ı değildir.

In [6]:
weather_impact = pd.read_csv(REPORTS_DIR / 'csv' / 'weather_demand_impact.csv')
display(weather_impact)

fig = px.bar(
    weather_impact,
    x='weather_condition',
    y='mean_total_demand_ratio',
    color='unit',
    barmode='group',
    text_auto='.3f',
    title='Bursa Gözlenen Hava Koşulu / Diğer Günler Talep Oranı',
)
fig.add_hline(y=1.0, line_dash='dash', line_color=PALETTE['rose'])
fig.update_xaxes(tickangle=30)
fig.update_layout(
    template='plotly_dark',
    paper_bgcolor='#111827',
    plot_bgcolor='#111827',
)
fig.show()

,unit,weather_condition,condition_days,comparison_days,condition_mean_total_demand,comparison_mean_total_demand,mean_total_demand_ratio,condition_mean_products_with_demand,comparison_mean_products_with_demand,observed_start,observed_end
0,KG,rainy_1mm_plus,426,870,298.130392,282.025638,1.057104,74.748826,70.539080,2023-01-01,2026-07-20
1,KG,heavy_rain_10mm_plus,110,1186,314.365927,284.810793,1.103771,77.945455,71.364250,2023-01-01,2026-07-20
2,KG,snow_day,68,1228,334.749412,284.692909,1.175826,79.441176,71.506515,2023-01-01,2026-07-20
3,KG,hot_30c_plus,245,1051,259.093347,293.899127,0.881572,65.563265,73.405328,2023-01-01,2026-07-20
4,KG,cold_below_10c,121,1175,321.295769,283.820480,1.132039,77.495868,71.348936,2023-01-01,2026-07-20
5,KG,sunny_dry,528,768,280.732163,291.848008,0.961912,69.066288,73.886719,2023-01-01,2026-07-20
6,ADT,rainy_1mm_plus,426,870,271.593897,251.511494,1.079847,71.718310,68.150575,2023-01-01,2026-07-20
7,ADT,heavy_rain_10mm_plus,110,1186,283.190909,255.786678,1.107137,73.600000,68.926644,2023-01-01,2026-07-20
8,ADT,snow_day,68,1228,303.088235,255.622150,1.185688,73.720588,69.079805,2023-01-01,2026-07-20
9,ADT,hot_30c_plus,245,1051,235.200000,263.453853,0.892756,68.261224,69.570885,2023-01-01,2026-07-20


### EDA karar ve handoff notu
Bu hücre EDA Expert tarafından çalıştırılmış grafiklerden sonra güncellenir:

- **Kanıt:** tarih kapsaması, eksik takvim günleri, ürün geçmişi ve birim tutarlılığı.
- **Karar:** satışsız günlerin `0` veya `missing_or_unobserved` politikası.
- **DataPrep'e aktarım:** `reports/csv/data_prep_recommendations.csv` ve `EDA_FINAL_REPORT.md`.
- **Risk:** kısa geçmişli/yeni ve aralıklı talep ürünleri ayrı etiketlenmelidir.

<div style="background:linear-gradient(90deg,#123524,#166534);border-left:6px solid #34d399;padding:18px 26px;border-radius:10px;margin:28px 0 12px;"><h2 style="color:#dcfce7;margin:0;">Bölüm 3 · Veri Hazırlama ve Zaman Güvenliği</h2><p style="color:#86efac;margin:7px 0 0;">Sorumlu agent: DataPrep Expert · Günlük panel, hedefler, feature'lar ve kronolojik split</p></div>

In [7]:
daily_panel_path = PROCESSED_DIR / 'daily_product_demand.csv'
split_path = MODEL_READY_DIR / 'split_metadata.json'
if daily_panel_path.exists():
    daily_panel = pd.read_csv(daily_panel_path)
    print('Günlük ürün-talep paneli yüklendi.')
    display(daily_panel.head())
else:
    print('DataPrep çıktısı henüz yok. DataPrep Expert paneli ve hedefleri ürettikten sonra bu hücreyi yeniden çalıştırın.')

if split_path.exists():
    split_metadata = json.loads(split_path.read_text(encoding='utf-8'))
    display(pd.DataFrame([split_metadata]).T.rename(columns={0: 'Değer'}))
else:
    print('split_metadata.json henüz yok; kronolojik split kararı bekleniyor.')

Günlük ürün-talep paneli yüklendi.


,date,product_id,product_name,unit,segment,current_status,first_sale,last_sale,store_observed,daily_demand,source_row_count,observation_status
0,2023-01-01,10002013,DANA KONTRAFILE BIFTEK KG,KG,continuous,active,2023-01-01,2026-07-20,1,0.506,1,observed_positive
1,2023-01-02,10002013,DANA KONTRAFILE BIFTEK KG,KG,continuous,active,2023-01-01,2026-07-20,1,0.000,0,observed_zero
2,2023-01-03,10002013,DANA KONTRAFILE BIFTEK KG,KG,continuous,active,2023-01-01,2026-07-20,1,1.082,1,observed_positive
3,2023-01-04,10002013,DANA KONTRAFILE BIFTEK KG,KG,continuous,active,2023-01-01,2026-07-20,1,0.876,1,observed_positive
4,2023-01-05,10002013,DANA KONTRAFILE BIFTEK KG,KG,continuous,active,2023-01-01,2026-07-20,1,0.624,1,observed_positive


,Değer
forecast_type,direct_multi_horizon_daily
forecast_origin_latest,2026-07-21
max_forecast_lead_days,180
lead_day_grid,"[1, 2, 3, 4, 5, 6, 7, 14, 21, 30, 45, 60, 90, ..."
split_basis,target_date
temporal_gap_days,7
splits,"{'train': {'start': '2023-01-30', 'end': '2026..."
feature_builder_version,3.0.0
calendar_version,TR_CALENDAR_2023_2027_V1
calendar_reference,data/reference/calendar_sources.json


### DataPrep karar ve handoff notu
Bu bölümde aşağıdakiler çalıştırılmış kanıtla görünmelidir: Türkçe miktar ayrıştırma sonucu, ürün-gün toplama, sıfır-talep politikası, seçili `target_date` için `demand_occurs` / `target_demand`, sadece forecast origin ve öncesinden üretilen snapshot feature'ları Türkiye resmî/dinî tatil ve MEB okul takvimi feature'ları ile target-date tabanlı train-validation-test zaman çizelgesi.

Ayrıntılı kaynak: `DATA_PREP_HANDOFF.md`, `feature_specification.json`, `split_metadata.json`.

<div style="background:linear-gradient(90deg,#3b0764,#581c87);border-left:6px solid #c084fc;padding:18px 26px;border-radius:10px;margin:28px 0 12px;"><h2 style="color:#f3e8ff;margin:0;">Bölüm 4 · Modelleme ve Rolling Backtest</h2><p style="color:#d8b4fe;margin:7px 0 0;">Sorumlu agent: Model Expert · Baseline, aday modeller, zaman uyumlu doğrulama</p></div>

In [8]:
comparison_path = REPORTS_DIR / 'csv' / 'model_comparison_results.csv'
backtest_path = REPORTS_DIR / 'csv' / 'backtest_results.csv'
for label, path in [('Model karşılaştırması', comparison_path), ('Rolling backtest', backtest_path)]:
    if path.exists():
        print(f'\n{label}: {path.name}')
        display(pd.read_csv(path).head(12))
    else:
        print(f'{label} çıktısı henüz yok: {path.name}')


Model karşılaştırması: model_comparison_results.csv


,unit,target,model,threshold,fit_seconds,pr_auc,precision,recall,f1,brier,positive_prediction_rate,mae,rmse,wape,bias
0,KG,occurrence,historical_frequency_baseline,0.36,0.000000,0.833511,0.718368,0.851449,0.779268,0.143619,0.525860,NaN,NaN,NaN,NaN
1,KG,occurrence,logistic_regression,0.51,0.499072,0.855611,0.734755,0.846566,0.786708,0.143074,0.511183,NaN,NaN,NaN,NaN
2,KG,occurrence,hist_gradient_boosting,0.47,3.305962,0.864258,0.732729,0.853812,0.788650,0.137072,0.516984,NaN,NaN,NaN,NaN
3,KG,occurrence,extra_trees,0.46,4.719333,0.864783,0.723634,0.865784,0.788353,0.137410,0.530822,NaN,NaN,NaN,NaN
4,KG,quantity_positive,rolling_mean_28_baseline,NaN,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,3.292792,24.290855,0.644926,-1.609506
5,KG,quantity_positive,hist_gradient_boosting_poisson,NaN,1.533428,NaN,NaN,NaN,NaN,NaN,NaN,2.961153,22.719570,0.579971,-0.802519
6,KG,quantity_positive,extra_trees,NaN,1.653677,NaN,NaN,NaN,NaN,NaN,NaN,3.037986,22.484537,0.595020,-0.757149
7,KG,end_to_end,extra_trees+hist_gradient_boosting_poisson,0.46,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.809554,16.195537,0.798839,-0.066625
8,ADT,occurrence,historical_frequency_baseline,0.29,0.000000,0.628189,0.524034,0.754026,0.618336,0.182904,0.494639,NaN,NaN,NaN,NaN
9,ADT,occurrence,logistic_regression,0.47,0.561713,0.677695,0.552450,0.747245,0.635250,0.192931,0.464977,NaN,NaN,NaN,NaN



Rolling backtest: backtest_results.csv


,unit,fold,train_end,validation_start,validation_end,train_rows,validation_rows,threshold,occurrence_pr_auc,occurrence_precision,occurrence_recall,occurrence_f1,occurrence_brier,occurrence_positive_prediction_rate,quantity_mae,quantity_rmse,quantity_wape,quantity_bias
0,KG,1,2025-11-11,2025-11-19,2025-12-30,159783,7403,0.47,0.855274,0.717485,0.871232,0.786919,0.133932,0.500608,1.317390,3.834600,0.763478,0.166145
1,KG,2,2026-01-06,2026-01-14,2026-02-24,169707,6062,0.53,0.872144,0.768240,0.811485,0.789271,0.133410,0.461234,1.646259,21.950252,0.754307,-0.124695
2,KG,3,2026-03-03,2026-03-11,2026-04-21,178138,7043,0.45,0.852626,0.708628,0.875000,0.783075,0.141934,0.541389,1.829005,10.213356,0.829992,0.053854
3,ADT,1,2025-11-11,2025-11-19,2025-12-30,178926,8180,0.50,0.723308,0.603672,0.737710,0.663994,0.185127,0.452812,1.274911,2.397984,1.015657,0.399169
4,ADT,2,2026-01-06,2026-01-14,2026-02-24,189787,7004,0.47,0.711932,0.585071,0.752429,0.658279,0.184759,0.472444,1.427852,3.451768,1.063449,0.437297
5,ADT,3,2026-03-03,2026-03-11,2026-04-21,199649,8713,0.49,0.676013,0.565913,0.711019,0.630221,0.182391,0.416160,1.185690,2.396650,1.182433,0.405176


### Özel takvim feature ablation

Aynı seçilmiş model ailesi, aynı train/validation döneminde iki kez ölçülür:
`full_calendar` 18 özel takvim alanını içerir; `special_calendar_removed` bunları
çıkarır ama standart gün/ay/hafta sonu alanlarını korur. Böylece takvimin gerçekten
genelleme performansına katkısı ayrı olarak görülür.

In [9]:
ablation = pd.read_csv(REPORTS_DIR / 'csv' / 'calendar_feature_ablation.csv')
display(ablation)

fig = px.bar(
    ablation,
    x='feature_set',
    y='occurrence_pr_auc',
    color='unit',
    barmode='group',
    text_auto='.4f',
    title='Özel Takvimli / Takvimsiz Validation PR-AUC',
)
fig.update_layout(
    template='plotly_dark',
    paper_bgcolor='#111827',
    plot_bgcolor='#111827',
)
fig.show()

,unit,feature_set,feature_count,occurrence_model,quantity_model,threshold,occurrence_pr_auc,occurrence_precision,occurrence_recall,occurrence_f1,occurrence_brier,occurrence_positive_prediction_rate,quantity_mae,quantity_rmse,quantity_wape,quantity_bias
0,KG,full_calendar,47,extra_trees,hist_gradient_boosting_poisson,0.46,0.864783,0.723634,0.865784,0.788353,0.137410,0.530822,1.809554,16.195537,0.798839,-0.066625
1,KG,special_calendar_removed,29,extra_trees,hist_gradient_boosting_poisson,0.49,0.863188,0.740155,0.840895,0.787316,0.137539,0.504054,1.801040,16.212078,0.795080,-0.140902
2,ADT,full_calendar,47,hist_gradient_boosting,hist_gradient_boosting_poisson,0.47,0.688724,0.557150,0.750297,0.639457,0.187051,0.462937,1.362536,3.209169,1.160987,0.500484
3,ADT,special_calendar_removed,29,hist_gradient_boosting,hist_gradient_boosting_poisson,0.49,0.686209,0.562159,0.732836,0.636250,0.189085,0.448135,1.335628,3.039960,1.138059,0.437230


### Hava feature ablation ve deployment kararı

Yedi time-safe Bursa iklim normali adayı aynı model ailesi ve aynı validation
döneminde eklenip çıkarılmıştır. KG'deki çok küçük fark diğer metriklerde ve ADT'de
tekrarlanmadığı için hava alanları final estimator feature listesinden çıkarılmıştır.
Bu karar hedef-gün hava sızıntısını ve zayıf sinyale overfitting'i önler; aday
alanlar model-ready veride ve EDA'da izlenebilir kalır.

In [10]:
weather_ablation = pd.read_csv(
    REPORTS_DIR / 'csv' / 'weather_feature_ablation.csv'
)
display(weather_ablation)

fig = px.bar(
    weather_ablation,
    x='feature_set',
    y='occurrence_pr_auc',
    color='unit',
    barmode='group',
    text_auto='.4f',
    title='Aday Hava Alanları Validation Ablation',
)
fig.update_layout(
    template='plotly_dark',
    paper_bgcolor='#111827',
    plot_bgcolor='#111827',
)
fig.show()

,unit,feature_set,feature_count,occurrence_model,quantity_model,threshold,occurrence_pr_auc,occurrence_precision,occurrence_recall,occurrence_f1,occurrence_brier,occurrence_positive_prediction_rate,quantity_mae,quantity_rmse,quantity_wape,quantity_bias
0,KG,candidate_with_weather,54,extra_trees,hist_gradient_boosting_poisson,0.47,0.864795,0.726970,0.860271,0.788023,0.137959,0.525021,1.810266,16.187587,0.799153,-0.046533
1,KG,deployed_weather_removed,47,extra_trees,hist_gradient_boosting_poisson,0.46,0.864783,0.723634,0.865784,0.788353,0.137410,0.530822,1.809554,16.195537,0.798839,-0.066625
2,ADT,candidate_with_weather,54,hist_gradient_boosting,hist_gradient_boosting_poisson,0.48,0.687859,0.560982,0.739956,0.638158,0.186854,0.453438,1.367107,3.189355,1.164882,0.513956
3,ADT,deployed_weather_removed,47,hist_gradient_boosting,hist_gradient_boosting_poisson,0.47,0.688724,0.557150,0.750297,0.639457,0.187051,0.462937,1.362536,3.209169,1.160987,0.500484


### Modelleme karar ve handoff notu
Model Expert bu alana; seçilen baseline üstünlüğünü, rolling backtest kararlılığını, occurrence için PR-AUC/eşik kararını, miktar için MAE-RMSE-WAPE/bias sonuçlarını özel takvim ablation sonucunu ve `KG`/`ADT` ile ürün/takvim segmenti kırılımını ekler. Test seti yalnızca nihai, donmuş seçimden sonra bir kez raporlanır.

Ayrıntılı kaynak: `MODEL_EVALUATION_REPORT.md` ve `MODEL_EXPERT_HANDOFF.md`.

<div style="background:linear-gradient(90deg,#3a2606,#713f12);border-left:6px solid #fbbf24;padding:18px 26px;border-radius:10px;margin:28px 0 12px;"><h2 style="color:#fef3c7;margin:0;">Bölüm 5 · Değerlendirme ve Model Kararı</h2><p style="color:#fcd34d;margin:7px 0 0;">Sorumlu agent: Model Expert · Birim, segment ve operasyonel risk değerlendirmesi</p></div>

In [11]:
metadata_path = BUNDLE_DIR / 'model_metadata.json'
if metadata_path.exists():
    model_metadata = json.loads(metadata_path.read_text(encoding='utf-8'))
    keys = ['model_version', 'forecast_type', 'forecast_origin', 'max_forecast_lead_days', 'units', 'model_layout', 'required_history_days', 'training_end_date', 'calendar_version', 'calendar_coverage', 'weather_feature_version', 'weather_location', 'weather_deployment_decision', 'occurrence_threshold', 'probability_calibrated', 'known_limitations']
    display(pd.DataFrame({'Alan': keys, 'Değer': [model_metadata.get(k) for k in keys]}))
    display(pd.DataFrame(model_metadata.get('unit_model_map', {})).T)
else:
    print("Model bundle henüz yok. Model Expert seçili modeli ve metadata'yı oluşturduğunda bu bölüm güncellenecek.")

,Alan,Değer
0,model_version,2026.07.26-weather-v3
1,forecast_type,direct_multi_horizon_daily
2,forecast_origin,2026-07-21
3,max_forecast_lead_days,180
4,units,"[KG, ADT]"
5,model_layout,per_unit
6,required_history_days,28
7,training_end_date,2026-04-21
8,calendar_version,TR_CALENDAR_2023_2027_V1
9,calendar_coverage,"{'start': '2023-01-01', 'end': '2027-12-31'}"


,occurrence_model_path,quantity_model_path,pipeline_path,occurrence_model_name,quantity_model_name
KG,occurrence_model_kg.pkl,quantity_model_kg.pkl,None,extra_trees,hist_gradient_boosting_poisson
ADT,occurrence_model_adt.pkl,quantity_model_adt.pkl,None,hist_gradient_boosting,hist_gradient_boosting_poisson


<div style="background:linear-gradient(90deg,#0c2b36,#155e75);border-left:6px solid #38bdf8;padding:18px 26px;border-radius:10px;margin:28px 0 12px;"><h2 style="color:#e0f2fe;margin:0;">Bölüm 6 · Deployment Simülasyonu ve İzleme</h2><p style="color:#7dd3fc;margin:7px 0 0;">Sorumlu agent: Deployment Expert · Bundle doğrulama, tahmin sözleşmesi ve gerçekleşen satış takibi</p></div>

In [12]:
required_bundle_files = ['model_metadata.json', 'product_catalog.csv']
bundle_check = pd.DataFrame({
    'Dosya': required_bundle_files,
    'Durum': [('Bulundu' if (BUNDLE_DIR / item).is_file() else 'Bekleniyor') for item in required_bundle_files]
})
display(bundle_check)

from app.services.forecast_service import DemandForecastService
service = DemandForecastService(PROJECT_ROOT)
options = service.product_options()
# Bu yalnızca örnektir; Streamlit aynı servise kullanıcının seçtiği tarihi gönderir.
example_date = pd.Timestamp('2027-01-02').date()
example_results = []
for unit in ['KG', 'ADT']:
    row = options[(options['unit'] == unit) & options['forecast_available'] & (options['current_status'] == 'active')].iloc[0]
    example_results.append(service.forecast(str(row['product_id']), example_date))
example_frame = pd.DataFrame(example_results)
example_frame['calendar_context_label'] = example_frame['calendar_context'].map(lambda value: ' · '.join(filter(None, [value.get('public_holiday_name'), value.get('religious_special_name'), value.get('school_status_label'), 'Ramazan' if value.get('is_ramadan') else None])))
example_frame['weather_source'] = example_frame['weather_context'].map(lambda value: value.get('source'))
example_frame['weather_used_by_model'] = example_frame['weather_context'].map(lambda value: value.get('used_by_model'))
display(example_frame[['product_name','unit','target_date','lead_days','calendar_context_label','weather_source','weather_used_by_model','demand_expected','demand_probability','display_quantity','status']])

display(Markdown('''**Deployment sonucu sözleşmesi:** `forecast_origin`, kullanıcının seçtiği `target_date`, `lead_days`, ürün, birim, talep olasılığı, günlük tahmin miktarı, model sürümü, veri tazeliği, `status` ve `warning_codes`.\n\nGösterilen miktar brüt mağaza talebidir. Mağaza stoku ve yoldaki sevkiyat sıfırsa başlangıç depo transfer miktarı olarak kullanılabilir; kesin net transfer için stok ve emniyet stoku bilgileri gerekir.'''))

,Dosya,Durum
0,model_metadata.json,Bulundu
1,product_catalog.csv,Bulundu


,product_name,unit,target_date,lead_days,calendar_context_label,weather_source,weather_used_by_model,demand_expected,demand_probability,display_quantity,status
0,$KOFTE HAMBURGER KG,KG,2027-01-02,165,Okul dönemi,time_safe_climatology,False,False,0.114443,0.0,forecast_ready_with_warning
1,$ERSAN JAMBON DANA 100 G (HAZIR DILIM),ADT,2027-01-02,165,Okul dönemi,time_safe_climatology,False,False,0.469675,0.0,forecast_ready_with_warning


**Deployment sonucu sözleşmesi:** `forecast_origin`, kullanıcının seçtiği `target_date`, `lead_days`, ürün, birim, talep olasılığı, günlük tahmin miktarı, model sürümü, veri tazeliği, `status` ve `warning_codes`.

Gösterilen miktar brüt mağaza talebidir. Mağaza stoku ve yoldaki sevkiyat sıfırsa başlangıç depo transfer miktarı olarak kullanılabilir; kesin net transfer için stok ve emniyet stoku bilgileri gerekir.

<div style="background:linear-gradient(135deg,#0f172a,#312e81);padding:30px;border-radius:18px;text-align:center;margin-top:36px;"><h3 style="color:#e0e7ff;margin:0;">Proje hikâyesi tamamlandı</h3><p style="color:#c7d2fe;margin:10px 0 0;">Bu çalıştırılmış notebook; ham veriden EDA, zaman güvenli DataPrep, rolling backtest, model kararı ve ürün + seçilen tarih Streamlit servisine kadar denetlenebilir zinciri tek yerde birleştirir.</p></div>